In [1]:
import librosa
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf
import os

In [2]:
import sys
from pathlib import Path

root_folder = Path().resolve().parent.parent
src_path = root_folder / 'src'
sys.path.append(str(src_path))

from tools import extract_audio
        

In [3]:
#annot_folder = "../data/annotation_xc/"
audio_folder = "./XC_BadenWurttem/"
length = 3 #three seconds of audio frames

In [4]:
audio_files = os.listdir(audio_folder)

In [5]:
annot_df = pd.read_csv("xc_BadenWurttem_filtered.csv")

In [6]:
annot_df.head(2)

,recording_id,mic,species,country,date,duration,user,recording_url
0,1165902,Sennheiser ME66,Dark Bush-cricket,Germany,2026-08-11,0:21,Frank Holzapfel,https://xeno-canto.org/1165902
1,1165901,Sennheiser ME66,NaN,Germany,2026-08-11,0:20,Frank Holzapfel,https://xeno-canto.org/1165901


NEGATIVES

In [7]:
output_folder= '../../data/test_data_5/audio/seed2/0/'

In [8]:
import os
import numpy as np
import librosa


segment_length = 3
first_start = 1
max_segments = 5

for _, row in annot_df.iterrows():

    audio_name = str(row["recording_id"])
    audio_path = os.path.join(audio_folder, audio_name + ".mp3")

    # Skip if file does not exist
    if not os.path.isfile(audio_path):
        print(f"File not found, skipping: {audio_path}")
        continue

    # Load audio at original sampling rate
    sig, rate = librosa.load(
        audio_path,
        sr=None,
        mono=True
    )

    # Actual duration in seconds
    duration = len(sig) / rate

    # Last possible 3-sec segment must end before file ends
    last_start = int(duration - segment_length)

    if last_start < first_start:
        print(f"Too short, skipping: {audio_name}")
        continue

    # Possible non-overlapping starts:
    # 1, 4, 7, 10, ...
    possible_starts = list(
        range(
            first_start,
            last_start + 1,
            segment_length
        )
    )

    # ---------------------------------------------------------
    # Select maximum 5 segments
    # ---------------------------------------------------------
    if len(possible_starts) <= max_segments:

        # Short recordings:
        # use all possible segments
        begin_times = possible_starts

    else:

        # Long recordings:
        # choose 5 uniformly distributed segments
        indices = np.linspace(
            0,
            len(possible_starts) - 1,
            max_segments,
            dtype=int
        )

        begin_times = [
            possible_starts[i]
            for i in indices
        ]

    print(
        f"{audio_name}: "
        f"duration={duration:.1f}s, "
        f"segments={begin_times}"
    )

    # Extract audio
    extract_audio(
        sig,
        rate,
        begin_times,
        output_folder,
        audio_name + ".***",
        segment_length
    )



1165902: duration=21.3s, segments=[1, 4, 7, 10, 16]
1165901: duration=20.2s, segments=[1, 4, 7, 10, 16]
1165900: duration=13.6s, segments=[1, 4, 7, 10]
1161563: duration=50.9s, segments=[1, 10, 22, 34, 46]
1161562: duration=16.0s, segments=[1, 4, 7, 10, 13]
1161561: duration=8.1s, segments=[1, 4]
1161560: duration=14.9s, segments=[1, 4, 7, 10]
1161559: duration=55.0s, segments=[1, 13, 25, 37, 52]
1161558: duration=49.1s, segments=[1, 10, 22, 34, 46]
1161557: duration=15.9s, segments=[1, 4, 7, 10]
1161555: duration=52.0s, segments=[1, 13, 25, 37, 49]
1161554: duration=9.0s, segments=[1, 4]
1157573: duration=41.4s, segments=[1, 10, 19, 28, 37]
1157572: duration=20.3s, segments=[1, 4, 7, 10, 16]
1099399: duration=31.8s, segments=[1, 7, 13, 19, 28]
1051878: duration=93.2s, segments=[1, 22, 43, 64, 88]
975837: duration=11.2s, segments=[1, 4, 7]
975832: duration=28.9s, segments=[1, 7, 13, 19, 25]
806870: duration=223.6s, segments=[1, 55, 109, 163, 220]
1172469: duration=9.9s, segments=[1, 4]

KeyboardInterrupt: 